In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain_core.tools import tool

@tool
def write_file_tool(filename: str, content: str) -> str:
    """파일을 지정된 경로에 작성합니다. (이름: write_file)"""
    return f"파일 '{filename}'에 {content}이 성공적으로 기록되었습니다."

@tool
def execute_sql_tool(query: str) -> str:
    """데이터베이스에서 SQL 쿼리를 실행합니다. (이름: execute_sql)"""
    return f"쿼리 '{query}'가 실행되었습니다. (영향을 받은 행: 1개)"

@tool
def read_data_tool(source: str) -> str:
    """지정된 소스에서 데이터를 읽어 옵니다. (이름: read_data)"""
    return f"'{source}'로부터 데이터를 성공적으로 불러왔습니다: [샘플 데이터]"

tools = [write_file_tool, execute_sql_tool, read_data_tool]

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=tools,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file_tool": True, # 승인, 수정, 거절
                "execute_sql_tool":  {
                    "allowed_decisions": ["approve", "reject"], # 승인, 수정
                },
                "read_data_tool": False, # 사용자 승인 없이 즉시 실행
            }
        ),
    ]
)

In [5]:
prompt = "abc 테이블의 모든 데이터를 삭제해줘."

In [6]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    config={"configurable": {"thread_id": "1"}}
)

In [7]:
res

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘.', additional_kwargs={}, response_metadata={}, id='ecafd0d8-c282-4442-9b09-565e76d896c8'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc;"}'}, '__gemini_function_call_thought_signatures__': {'5ade34ae-f905-4624-9d06-1ac364a136aa': 'EtsGCtgGAb4+9vuQEhtFLxCcYSX4ESsXnWA5RLuAe++n5JISKlc6vUl1XeyNXSeldgUGiJpvwy5I6zUXnzS1rQvqphCN6flIp4v8mtBeoxLQoWlUwfe0jT+4cUGUeYVttDCV/GBT7rzmq0IVQYXgKLrnAz0sn1zs6f6l0hUcg4D3/NbTPndxdUFguX/SM8ElVrCo4NseZoy1v9OhtSO+3pUPunV9uJ8+DjHjHnmy27JYj4TI7Dvlh1dAlC67QoiQaqedXyx+Il5/rmd09GME+l9GttK5t5uZgXnrNeROdY9c7fpD8Bv0iyBIFplSzFCQwIFjubRcbvEOqv84gAvmJjNUaegGCg8YfMNkppP3c130UUsgKXM/AruRJDbK+ojY86z94qWpc0duS+Ps2tAf+wUd8sa6ib/FPIxbiUIeY5WKnZtbqF4VVm+72oTVl80644yo2mNXs3zgVw2GAvzr0p0iIuS90cKZtOu7u53gaiBqRDSnYduZL55VC4/fXIqtPRmSKeJ+ZQa1d9jWRk5H8gSj95ETINHItZgruqPuip+CXdv8XlrxmwVK2V9l0PiZqOJjcvVS1fVo4XTAJiqe33LiKbev8iUa7Aq3yPASJKqmncS9F+

In [8]:
res['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc;'}, 'description': "Tool execution requires approval\n\nTool: execute_sql_tool\nArgs: {'query': 'DELETE FROM abc;'}"}], 'review_configs': [{'action_name': 'execute_sql_tool', 'allowed_decisions': ['approve', 'reject']}]}, id='169ff78f163f705eb906f9b41b535ceb')]

In [9]:
from langgraph.types import Command

agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config={"configurable": {"thread_id": "1"}}
)

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘.', additional_kwargs={}, response_metadata={}, id='ecafd0d8-c282-4442-9b09-565e76d896c8'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc;"}'}, '__gemini_function_call_thought_signatures__': {'5ade34ae-f905-4624-9d06-1ac364a136aa': 'EtsGCtgGAb4+9vuQEhtFLxCcYSX4ESsXnWA5RLuAe++n5JISKlc6vUl1XeyNXSeldgUGiJpvwy5I6zUXnzS1rQvqphCN6flIp4v8mtBeoxLQoWlUwfe0jT+4cUGUeYVttDCV/GBT7rzmq0IVQYXgKLrnAz0sn1zs6f6l0hUcg4D3/NbTPndxdUFguX/SM8ElVrCo4NseZoy1v9OhtSO+3pUPunV9uJ8+DjHjHnmy27JYj4TI7Dvlh1dAlC67QoiQaqedXyx+Il5/rmd09GME+l9GttK5t5uZgXnrNeROdY9c7fpD8Bv0iyBIFplSzFCQwIFjubRcbvEOqv84gAvmJjNUaegGCg8YfMNkppP3c130UUsgKXM/AruRJDbK+ojY86z94qWpc0duS+Ps2tAf+wUd8sa6ib/FPIxbiUIeY5WKnZtbqF4VVm+72oTVl80644yo2mNXs3zgVw2GAvzr0p0iIuS90cKZtOu7u53gaiBqRDSnYduZL55VC4/fXIqtPRmSKeJ+ZQa1d9jWRk5H8gSj95ETINHItZgruqPuip+CXdv8XlrxmwVK2V9l0PiZqOJjcvVS1fVo4XTAJiqe33LiKbev8iUa7Aq3yPASJKqmncS9F+